# 🏛️ NLP Cuối Kỳ – Chatbot Hỏi Đáp Pháp Luật ĐẠI CƯƠNG DÀNH CHO SINH VIÊN ĐẠI HỌC MỞ TP.HCM

**Dataset:** 715 câu hỏi, 10 lớp | 299 đoạn RAG  
**Models:** PhoBERT Classifier + multilingual-e5 Dense RAG

---

### 📋 Nội dung
1. Cài đặt & Clone repo
2. RAG Pipeline (PhoBERT + E5)
3. End-to-End Test (20 câu)
4. Demo tương tác

## 1. Cài đặt

In [ ]:
# Cài dependencies mới cho pipeline nâng cấp
!pip install sentence-transformers faiss-cpu rank-bm25 underthesea pandas numpy -q

# Thay link GitHub của nhóm:
# !git clone https://github.com/YOUR_USERNAME/nlp-phapluat.git
# %cd nlp-phapluat

print('✓ Cài đặt xong')
print('  - sentence-transformers : encode với multilingual-e5')
print('  - faiss-cpu             : vector search index')

## 2. RAG Pipeline (PhoBERT + multilingual-e5)

> **Lưu ý:** Lần đầu chạy sẽ download E5 model (~500MB). Các lần sau load từ cache nhanh hơn.

In [ ]:
from rag.rag_pipeline import RAGPipeline

rag = RAGPipeline(
    rag_data_path='data/rag_data.json',
    phobert_model_path='model/phobert_classifier',
    top_k=3,
)
print(f"Pipeline đã sẵn sàng với {len(rag.all_docs)} đoạn và {len(rag.chapters)} chương.")

In [ ]:
# Test nhanh 3 câu
for q in ['Pháp luật là gì', 'Quy phạm pháp luật là gì', 'Vi phạm pháp luật là gì']:
    rag.answer(q, verbose=True)

## 3. End-to-End Test (20 câu)

In [ ]:
from app.pipeline import answer_dict

questions_kw = [
    ('Pháp luật là gì',                             ['pháp luật', 'quy tắc']),
    ('Quy phạm pháp luật là gì',                    ['quy phạm', 'chế tài', 'giả định']),
    ('Quan hệ pháp luật là gì',                     ['quan hệ', 'chủ thể']),
    ('Nhà nước là gì',                              ['nhà nước', 'quyền lực']),
    ('Bộ máy nhà nước gồm những cơ quan nào',       ['bộ máy', 'cơ quan']),
    ('Vi phạm pháp luật gồm những yếu tố nào',     ['vi phạm', 'cấu thành']),
    ('Trách nhiệm pháp lý là gì',                   ['trách nhiệm', 'pháp lý']),
    ('Thực hiện pháp luật là gì',                   ['thực hiện', 'pháp luật']),
    ('Áp dụng pháp luật là gì',                     ['áp dụng', 'pháp luật']),
    ('Hệ thống pháp luật Việt Nam gồm gì',          ['hệ thống', 'ngành luật']),
    ('Chức năng của nhà nước là gì',                ['chức năng', 'nhà nước']),
    ('Hình thức pháp luật là gì',                   ['hình thức', 'pháp luật']),
    ('Tập quán pháp là gì',                         ['tập quán']),
    ('Sự kiện pháp lý là gì',                       ['sự kiện', 'pháp lý']),
    ('Năng lực pháp luật là gì',                    ['năng lực']),
    ('Chế tài trong quy phạm pháp luật là gì',      ['chế tài']),
    ('Ngành luật là gì',                            ['ngành luật']),
    ('Nhà nước xuất hiện từ khi nào',               ['nhà nước', 'nguồn gốc']),
    ('Bản chất của pháp luật là gì',                ['bản chất', 'pháp luật']),
    ('Trách nhiệm hình sự là gì',                   ['trách nhiệm', 'hình sự']),
]

hits = 0
for q, kws in questions_kw:
    d = answer_dict(q, top_k=3)
    top_content = d['passages'][0]['content'].lower() if d['passages'] else ''
    hit = any(kw in top_content for kw in kws)
    if hit: hits += 1
    mode = d.get('route_mode', '?')[:6]
    conf = d.get('confidence', 0.0)
    print(f"  {'✓' if hit else '✗'}  [{mode}|{conf:.2f}]  {q}")

print(f'\n🏆 Kết quả: {hits}/{len(questions_kw)} = {hits/len(questions_kw):.2%}')

## 4. Demo tương tác

In [ ]:
from app.pipeline import answer

# ← Thay câu hỏi tại đây!
my_question = 'Quy phạm pháp luật là gì'
print(answer(my_question, top_k=3))

In [ ]:
# Vòng lặp hỏi đáp tương tác
while True:
    q = input('❓ Nhập câu hỏi (quit để thoát): ').strip()
    if q.lower() in ('quit', 'thoat', 'q', ''): break
    print(answer(q))
    print()